# 1. Setup
**Baseline 2: Multi-task CNN.** One shared backbone with two heads, one for crop and one for disease.
The shared features must serve both tasks. A pair score is log P(crop) + log P(disease).

In [ ]:
import copy
import json
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # lets the notebook import common/

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import config
from common.config import CHECKPOINT_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, set_seed
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

MODEL_NAME = "baseline2_multitask"
DEBUG = False  # Full training run.
HP = {"backbone": "resnet50", "lr": 1e-4, "weight_decay": 1e-4, "epochs": 20,
      "batch_size": 64, "optimizer": "AdamW", "label_smoothing": 0.1, "disease_loss_weight": 1.0}
EPOCHS = 1 if DEBUG else HP["epochs"]
config.NUM_WORKERS = 4
HP["batch_size"] = 16 if DEBUG else HP["batch_size"]
HP["debug"] = DEBUG
RUN_NAME = MODEL_NAME + ("_debug" if DEBUG else "")
set_seed(42)
OUTPUT_DIR = Path.cwd() / "baseline2_multitask_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR = OUTPUT_DIR  # shared evaluation saves metrics and confusion files here
print(f"Saving results to {OUTPUT_DIR.resolve()}")


# 2. Data

In [ ]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="imagenet", debug=DEBUG)
# Reduce host memory pressure before any loader is iterated.
for loader in (train_loader, val_loader, seen_loader, unseen_loader):
    loader.pin_memory = False
pairs = get_pairs().to(DEVICE)

# 3. Model

In [ ]:
class MultiTaskNet(nn.Module):
    """Shared backbone, two linear heads."""

    def __init__(self, backbone_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


model = MultiTaskNet(HP["backbone"]).to(DEVICE)


@torch.no_grad()
def predict(images):
    model.eval()
    crop_logits, disease_logits = model(images.to(DEVICE))
    crop_log_prob = F.log_softmax(crop_logits.float(), dim=1)
    disease_log_prob = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log_prob[:, pairs[:, 0]] + disease_log_prob[:, pairs[:, 1]]

# 4. Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=HP["label_smoothing"])
best_val, best_state = -1.0, None
history = []

for epoch in range(EPOCHS):
    model.train()
    crop_sum = disease_sum = total_sum = example_count = 0
    learning_rate = optimizer.param_groups[0]["lr"]
    for images, crop_labels, disease_labels, _ in train_loader:
        images = images.to(DEVICE)
        crop_labels, disease_labels = crop_labels.to(DEVICE), disease_labels.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()):
            crop_logits, disease_logits = model(images)
            crop_loss = criterion(crop_logits, crop_labels)
            disease_loss = criterion(disease_logits, disease_labels)
            loss = crop_loss + HP["disease_loss_weight"] * disease_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        count = images.size(0)
        crop_sum += crop_loss.item() * count
        disease_sum += disease_loss.item() * count
        total_sum += loss.item() * count
        example_count += count
    scheduler.step()
    val_top1 = val_accuracy(predict, val_loader)
    row = {"epoch": epoch + 1, "train_crop_loss": crop_sum / example_count,
           "train_disease_loss": disease_sum / example_count,
           "train_total_loss": total_sum / example_count,
           "val_pair_top1_percent": val_top1, "learning_rate": learning_rate}
    history.append(row)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
    if val_top1 > best_val:  # keep the best epoch by val, never by test
        best_val, best_state = val_top1, copy.deepcopy(model.state_dict())

    x = [item["epoch"] for item in history]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for key, label in (("train_crop_loss", "Crop"), ("train_disease_loss", "Disease"),
                       ("train_total_loss", "Weighted total")):
        axes[0].plot(x, [item[key] for item in history], marker="o", label=label)
    axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Mean loss")
    axes[0].legend()
    axes[1].plot(x, [item["val_pair_top1_percent"] for item in history], marker="o")
    axes[1].set(title="Validation pair top-1", xlabel="Epoch", ylabel="Accuracy (%)", ylim=(0, 100))
    axes[2].plot(x, [item["learning_rate"] for item in history], marker="o")
    axes[2].set(title="Learning rate", xlabel="Epoch", ylabel="Rate")
    for axis in axes:
        axis.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "training_progress.png", dpi=150, bbox_inches="tight")
    clear_output(wait=True)
    print(f"Epoch {epoch + 1}/{EPOCHS}: crop loss {row['train_crop_loss']:.3f}, "
          f"disease loss {row['train_disease_loss']:.3f}, val top-1 {val_top1:.2f}% "
          f"(best {best_val:.2f}%)")
    display(fig)
    plt.close(fig)

model.load_state_dict(best_state)

# 5. Evaluation

In [ ]:
results = evaluate(predict, seen_loader, unseen_loader, RUN_NAME, HP, robust=True)

# 6. Save

In [ ]:
torch.save({"model_name": RUN_NAME, "state_dict": model.state_dict(), "hp": HP,
            "best_val_top1_percent": best_val}, OUTPUT_DIR / f"{RUN_NAME}.pt")
(OUTPUT_DIR / "run_summary.json").write_text(
    json.dumps({"model": RUN_NAME, "best_val_top1_percent": best_val,
                "best_epoch": max(history, key=lambda row: row["val_pair_top1_percent"])["epoch"],
                "results": results}, indent=2), encoding="utf-8")
print(f"Saved results to {OUTPUT_DIR.resolve()}")
